## Real angular moments in the Wigner basis for B⁺ → K⁺π⁺π⁻ℓ⁻ℓ⁺

This notebook starts from the squared amplitude saved by `AmpiezzaWigner.ipynb` and applies two corrections.

**1. Non-resonant amplitude.** In `AmpiezzaWigner.ipynb` the non-resonant term is $a_{NR}\,D^0 D^0 d^0_{00}(\theta_\ell)$:
constant also in $\theta_\ell$, without leptonic amplitude and identical for the two chiralities.
For massless leptons the $\ell^-\ell^+$ pair always has projection $\lambda_{\ell^-}-\lambda_{\ell^+}=\eta=\mp1$ along its axis,
so every amplitude must contain $d^1_{\lambda,\eta}(\theta_\ell)$; a term constant in $\theta_\ell$ generates interferences with leptonic index $k\neq0$,
which break the orthogonality of the basis. Here the non-resonant term is written as the $K(1460)$ channel without propagator:
Kππ in S-wave ($J=0$, $\lambda=0$, $L_B=1$), hadronic part constant over phase space, produced by the weak current:
$$\mathcal{A}^\eta_{NR} = a_{NR}\; B_1(q_B)\; H(\eta,0,0,1,\mathrm{NR})\; L^\eta\; d^{1*}_{0,\eta}(\theta_\ell)\,\frac{1}{4\pi},\qquad L^\eta = 2\sqrt{2}\,q .$$
The correction is applied directly to the saved squared amplitude, replacing $a_{NR}\to a_{NR}F_\eta$ with $\eta$ read from the other factors of each term,
and the Clebsch-Gordan reduction is then redone.

**2. Real basis.** After the Clebsch-Gordan series the squared amplitude is
$$|\mathcal{A}|^2=\sum_k A_k\,\mathcal{D}_k(\Omega),\qquad \mathcal{D}_k = d^{j_\ell}_{m_\ell,k_\ell}(\theta_\ell)\,D^{j_1}_{m_1,k_1}(\phi,\theta_1,0)\,D^{j_2}_{m_2,k_2}(\chi,\theta_2,0).$$
Every product $\mathcal{D}$ has a partner $\mathcal{D}'$ with all indices reversed in sign, and $\mathcal{D}^*=p\,\mathcal{D}'$ with $p=(-1)^{\sum(m-k)}$.
Since $|\mathcal{A}|^2$ is real, the coefficient of the partner is $A'=p\,A^*$, hence
$$A\,\mathcal{D}+A'\,\mathcal{D}' = 2\,\mathrm{Re}(A\,\mathcal{D}) = \underbrace{2\,\mathrm{Re}A}_{\text{moment}}\;\mathrm{Re}\,\mathcal{D}\;+\;\underbrace{(-2\,\mathrm{Im}A)}_{\text{moment}}\;\mathrm{Im}\,\mathcal{D}.$$
Each pair therefore gives **two** real functions, $\mathrm{Re}\,\mathcal{D}$ and $\mathrm{Im}\,\mathcal{D}$, each with its own real moment.
If $\mathcal{D}=\mathcal{D}'$ (all indices zero) the function is already real and its moment is $A$, which must be real.

The angular functions are kept in terms of Wigner matrices, without expanding them into sines and cosines.
**Not** addressed here: the Blatt-Weisskopf radius $r_3$ used instead of $r_2$ in the one-res channel, and the Clebsch-Gordan convention for $L_B$.

### Setup

In [1]:
import os, time, random
from collections import defaultdict
import dill
import numpy as np
import sympy as sp
from sympy.physics.quantum.spin import Rotation as rot, WignerD
from sympy.physics.wigner import clebsch_gordan
from sympy.core.function import AppliedUndef
from tqdm.auto import tqdm

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=160)

# Cartella di lavoro: quella che contiene i .pkl prodotti da AmpiezzaWigner.ipynb
WORKDIR = os.getcwd()
OUTDIR = os.path.join(WORKDIR, "Moments_ReIm")
os.makedirs(OUTDIR, exist_ok=True)

phi, chi, theta1, theta2, thetal = sp.symbols('phi chi theta1 theta2 thetal', real=True)
HADR = {'H', 'Hbar', 'G'}

### Numerical evaluator
The checks require fast evaluation of very large expressions. Random values are assigned to all dynamical parameters:
complex for $a_i$, $H$, $\bar H$, $G$, $R$; positive real for the Blatt-Weisskopf factors, $q$ and the radii.
The evaluator walks the expression tree instead of using `subs`, which would be too slow.

In [2]:
def is_real_param(a):
    if isinstance(a, AppliedUndef):
        return a.func.__name__ == 'BW'
    return bool(a.is_real)

def param_atoms(expr):
    # parametri dinamici: funzioni applicate (H, R, BW, ...) e simboli liberi, esclusi gli angoli
    out = {a for a in expr.atoms(AppliedUndef)} | set(expr.free_symbols)
    return out - {phi, chi, theta1, theta2, thetal}

def draw_values(seed, atoms):
    rng = random.Random(seed)
    vals = {}
    for a in sorted(atoms, key=str):
        if is_real_param(a):
            vals[a] = rng.uniform(0.5, 1.5)
        else:
            vals[a] = complex(rng.uniform(-1, 1), rng.uniform(-1, 1))
    return vals

def random_angles(rng):
    return {phi: rng.uniform(0, 2*np.pi), chi: rng.uniform(0, 2*np.pi),
            theta1: rng.uniform(0, np.pi), theta2: rng.uniform(0, np.pi), thetal: rng.uniform(0, np.pi)}

_num_cache, _wig_cache = {}, {}
def fval(f, V, ang=None):
    # Valore numerico complesso di f, dati i parametri V e (opzionale) gli angoli ang
    if f.is_number:
        r = _num_cache.get(f)
        if r is None:
            r = _num_cache[f] = complex(sp.N(f))
        return r
    if f in V:
        return complex(V[f])
    if f.is_Mul:
        r = 1
        for g in f.args:
            r *= fval(g, V, ang)
        return r
    if f.is_Add:
        return sum(fval(g, V, ang) for g in f.args)
    if f.is_Pow:
        return fval(f.base, V, ang) ** int(f.exp)
    if isinstance(f, sp.conjugate):
        return fval(f.args[0], V, ang).conjugate()
    if isinstance(f, sp.re):
        return complex(fval(f.args[0], V, ang).real)
    if isinstance(f, sp.im):
        return complex(fval(f.args[0], V, ang).imag)
    if isinstance(f, sp.Abs):
        return complex(abs(fval(f.args[0], V, ang)))
    if isinstance(f, WignerD):
        key = (f, tuple(sorted(ang.items(), key=str)))
        r = _wig_cache.get(key)
        if r is None:
            r = _wig_cache[key] = complex(sp.N(f.doit().subs(ang)))
        return r
    raise TypeError(f"Oggetto non valutabile: {f} ({type(f)})")

### Loading the saved squared amplitude
`WIGAtotsq_terms*.pkl` contains the expanded $|\mathcal{A}|^2=|\mathcal{A}^{L}|^2+|\mathcal{A}^{R}|^2$, with the Wigner matrices not yet reduced.

In [3]:
t0 = time.time()
atotsq_file = sorted(f for f in os.listdir(WORKDIR) if f.startswith("WIGAtotsq_terms") and f.endswith(".pkl"))[-1]
with open(atotsq_file, "rb") as f:
    Atotsq_old = dill.load(f)["Atotsq"]
terms_old = sp.Add.make_args(Atotsq_old)
print(f"Termini nel modulo quadro ({atotsq_file}): {len(terms_old)}")
print(f"Tempo di caricamento: {time.time()-t0:.1f} s")

Termini nel modulo quadro (WIGAtotsq_terms33281.pkl): 33281
Tempo di caricamento: 4.0 s


### Correction of the non-resonant term
In the squared amplitude $a_{NR}$ appears only as a bare factor (its three $D^0$ are equal to 1).
For each term containing $a_{NR}$ the chirality $\eta$ is read from the hadronic amplitudes $H,\bar H,G$ present, and the replacement
$a_{NR}\to a_{NR}F_\eta$ (hence $a_{NR}^*\to a_{NR}^*F_\eta^*$) is applied, with
$$F_\eta = B_1(q_B)\,H(\eta,0,0,1,\mathrm{NR})\;2\sqrt2\,q\;d^{1*}_{0,\eta}(\theta_\ell)\,\frac{1}{4\pi}.$$
The term $|a_{NR}|^2$ carries no $\eta$: it appears with a factor 2, once per chirality, and is split into the two contributions $\eta=\mp1$.

**Independent check.** The new non-resonant term has the same structure as the $K(1460)$ channel (coefficient $a_1$) with propagator $R=1$.
Therefore the terms with $a_{NR}$ and without $a_1$ of the new squared amplitude (excluding those with $a_{17}, a_{18}$, which also contain $K(1460)$) must coincide with the terms with $a_1$ and without $a_{NR}$ of the old one,
after the replacement $a_1\to a_{NR}$, $R(K(1460))\to1$, $H(\eta,0,0,1,K(1460))\to H(\eta,0,0,1,\mathrm{NR})$.

In [4]:
nr_file = "NR_WIGAtotsq.pkl"
syms = {s.name: s for s in Atotsq_old.free_symbols}
aNR, a1, q = syms['a_NR'], syms['a1'], syms['q']
H_func = next(a.func for a in Atotsq_old.atoms(AppliedUndef) if a.func.__name__ == 'H')
R_func = next(a.func for a in Atotsq_old.atoms(AppliedUndef) if a.func.__name__ == 'R')
BW_B = next(a for a in Atotsq_old.atoms(AppliedUndef)
            if a.func.__name__ == 'BW' and a.args[0] == 1 and a.args[1].name == 'rB')
NR = sp.Symbol('NR')
K1460 = sp.Symbol('K(1460)')

def F_nr(eta):
    return BW_B * H_func(eta, 0, 0, 1, NR) * 2 * sp.sqrt(2) * q * sp.conjugate(rot.D(1, 0, eta, 0, thetal, 0)) / (4 * sp.pi)

def term_eta(t):
    etas = {a.args[0] for a in t.atoms(AppliedUndef) if a.func.__name__ in HADR}
    if not etas:
        return None
    assert len(etas) == 1, t
    return int(list(etas)[0])

if os.path.exists(nr_file):
    with open(nr_file, "rb") as f:
        Atotsq = dill.load(f)
    print(f"Caricato {nr_file}")
else:
    new_terms, n_nr = [], 0
    for t in tqdm(terms_old, desc="Correzione a_NR"):
        if not t.has(aNR):
            new_terms.append(t)
            continue
        n_nr += 1
        e = term_eta(t)
        for eta, w in ([(e, 1)] if e is not None else [(-1, sp.Rational(1, 2)), (1, sp.Rational(1, 2))]):
            new_terms.append(w * t.xreplace({aNR: aNR * F_nr(eta)}))
    Atotsq = sp.Add(*new_terms)
    with open(nr_file, "wb") as f:
        dill.dump(Atotsq, f)
    print(f"Termini con a_NR corretti: {n_nr}")
terms_new = sp.Add.make_args(Atotsq)
print(f"Termini nel nuovo modulo quadro: {len(terms_new)}")

# --- Controllo indipendente con il canale K(1460) ---
sub_k = {a1: aNR}
for a in Atotsq_old.atoms(AppliedUndef):
    if a.func.__name__ == 'R' and a.args[0] == K1460:
        sub_k[a] = sp.S(1)
    if a.func.__name__ == 'H' and a.args[-1] == K1460:
        sub_k[a] = H_func(*a.args[:-1], NR)
# si escludono i termini con a17, a18 (canali a due risonanze che contengono anch'essi K(1460))
excl = [syms['a17'], syms['a18']]
old_a1 = sp.Add(*[t for t in terms_old if t.has(a1) and not t.has(aNR) and not any(t.has(x) for x in excl)]).xreplace(sub_k)
new_nr = sp.Add(*[t for t in terms_new if t.has(aNR) and not t.has(a1) and not any(t.has(x) for x in excl)])
chk_atoms = param_atoms(old_a1) | param_atoms(new_nr)
rng = random.Random(3)
max_chk = 0.0
for seed in (1, 2, 3):
    V = draw_values(seed, chk_atoms)
    ang = random_angles(rng)
    x, y = fval(old_a1, V, ang), fval(new_nr, V, ang)
    max_chk = max(max_chk, abs(x - y) / abs(x))
    print(f"K(1460) con R=1: {x.real: .10e}   nuovo NR: {y.real: .10e}   scarto rel. = {abs(x-y)/abs(x):.1e}")
assert max_chk < 1e-10

# positività del nuovo |A|^2
all_atoms = param_atoms(Atotsq)
min_val = min(fval(Atotsq, draw_values(s, all_atoms), random_angles(rng)).real for s in range(4, 8))
print(f"Minimo di |A|^2 su 4 punti casuali: {min_val:.4e} (deve essere > 0)")
assert min_val > 0

Caricato NR_WIGAtotsq.pkl
Termini nel nuovo modulo quadro: 33282


K(1460) con R=1:  5.6192318572e-02   nuovo NR:  5.6192318572e-02   scarto rel. = 2.5e-16


K(1460) con R=1: -5.8633188110e-03   nuovo NR: -5.8633188110e-03   scarto rel. = 0.0e+00


K(1460) con R=1: -7.5656371169e-04   nuovo NR: -7.5656371169e-04   scarto rel. = 2.3e-15


Minimo di |A|^2 su 4 punti casuali: 1.5048e-02 (deve essere > 0)


### Reduction with the Clebsch-Gordan series
In each term the conjugates are resolved, $D^{J*}_{M,K}=(-1)^{M-K}D^J_{-M,-K}$, and for each group of angles
($\theta_\ell$; $\phi,\theta_1$; $\chi,\theta_2$) the product of Wigner matrices is reduced with the Clebsch-Gordan series
$$D^{j_1}_{m_1k_1}D^{j_2}_{m_2k_2}=\sum_J\langle j_1m_1j_2m_2|JM\rangle\langle j_1k_1j_2k_2|JK\rangle D^J_{MK},$$
applied repeatedly when there are more than two factors (here the $\theta_\ell$ group can have three, one of which is $D^0=1$).
The result is saved in `NR_angular_trig_coefficients.pkl`.

In [5]:
cg_file = "NR_angular_trig_coefficients.pkl"

def angle_group(D):
    ang = D.args[3:]
    return 'l' if thetal in ang else ('1' if theta1 in ang else '2')

GROUP_ANGLES = {'l': (0, thetal, 0), '1': (phi, theta1, 0), '2': (chi, theta2, 0)}

_cg_cache = {}
def reduce_group(factors):
    # factors: tupla di (J, M, K); restituisce {(J, M, K): coefficiente}
    key = tuple(sorted(factors))
    if key in _cg_cache:
        return _cg_cache[key]
    cur = {(0, 0, 0): sp.S(1)}
    for (j, m, k) in key:
        nxt = defaultdict(lambda: sp.S(0))
        for (J, M, K), c in cur.items():
            for Jn in range(abs(J - j), J + j + 1):
                if abs(M + m) > Jn or abs(K + k) > Jn:
                    continue
                cgM = clebsch_gordan(J, j, Jn, M, m, M + m)
                if cgM == 0:
                    continue
                cgK = clebsch_gordan(J, j, Jn, K, k, K + k)
                if cgK == 0:
                    continue
                nxt[(Jn, M + m, K + k)] += c * cgM * cgK
        cur = {kk: v for kk, v in nxt.items() if v != 0}
    _cg_cache[key] = cur
    return cur

if os.path.exists(cg_file):
    with open(cg_file, "rb") as f:
        angular_trig_coefficients = dill.load(f)
    print(f"Caricato {cg_file}")
else:
    t0 = time.time()
    # 1. Raggruppa i termini per contenuto di Wigner (dopo aver risolto i coniugati)
    buckets = defaultdict(list)
    for t in tqdm(terms_new, desc="Coniugati"):
        num, groups, phase = [], {'l': [], '1': [], '2': []}, 1
        for f in sp.Mul.make_args(t):
            base, ex = (f.base, int(f.exp)) if f.is_Pow else (f, 1)
            conj = isinstance(base, sp.conjugate) and isinstance(base.args[0], WignerD)
            inner = base.args[0] if conj else base
            if isinstance(inner, WignerD):
                J, M, K = (int(x) for x in inner.args[:3])
                if conj:
                    phase *= (-1) ** ((M - K) * ex)
                    M, K = -M, -K
                groups[angle_group(inner)] += [(J, M, K)] * ex
            else:
                num.append(f)
        wkey = tuple(tuple(sorted(groups[g])) for g in ('l', '1', '2'))
        buckets[wkey].append(phase * sp.Mul(*num))
    print(f"Prodotti di Wigner distinti prima della riduzione: {len(buckets)}")

    # 2. Serie di Clebsch-Gordan e accumulo per prodotto ridotto D_l D_1 D_2
    acc = defaultdict(list)
    for wkey, nums in tqdm(buckets.items(), desc="Clebsch-Gordan"):
        red = [reduce_group(g) for g in wkey]
        for kl, cl in red[0].items():
            for k1, c1 in red[1].items():
                for k2, c2 in red[2].items():
                    c = cl * c1 * c2
                    DDD = (rot.D(*kl, *GROUP_ANGLES['l']) * rot.D(*k1, *GROUP_ANGLES['1'])
                           * rot.D(*k2, *GROUP_ANGLES['2']))
                    acc[DDD].extend(c * n for n in nums)
    angular_trig_coefficients = {}
    for DDD, lst in tqdm(acc.items(), desc="Somma coefficienti"):
        v = sp.Add(*lst)
        if v != 0:
            angular_trig_coefficients[DDD] = v
    with open(cg_file, "wb") as f:
        dill.dump(angular_trig_coefficients, f)
    print(f"Tempo: {time.time()-t0:.1f} s")

angular_basis = sorted(angular_trig_coefficients, key=lambda x: (len(str(x)), str(x)))
print(f"Prodotti complessi di Wigner D dopo la riduzione: {len(angular_basis)}")
print(f"Termini totali nei coefficienti: {sum(len(sp.Add.make_args(v)) for v in angular_trig_coefficients.values())}")
lep_k = [D for D in angular_basis if any(angle_group(d) == 'l' and d.args[2] != 0 for d in D.as_ordered_factors())]
print(f"Prodotti con indice leptonico k != 0: {len(lep_k)}")

Caricato NR_angular_trig_coefficients.pkl
Prodotti complessi di Wigner D dopo la riduzione: 461
Termini totali nei coefficienti: 217340
Prodotti con indice leptonico k != 0: 0


### Numerical values of the coefficients

In [6]:
dyn_atoms = set()
for v in angular_trig_coefficients.values():
    dyn_atoms |= param_atoms(v)
V1, V2 = draw_values(1, dyn_atoms), draw_values(2, dyn_atoms)
t0 = time.time()
num1 = {k: fval(v, V1) for k, v in angular_trig_coefficients.items()}
num2 = {k: fval(v, V2) for k, v in angular_trig_coefficients.items()}
print(f"Valutazione numerica dei coefficienti: {time.time()-t0:.1f} s")

Valutazione numerica dei coefficienti: 4.2 s


### Pairing $\mathcal{D}\leftrightarrow\mathcal{D}'$
For each product the partner $\mathcal{D}'$ and the phase $p$ are constructed.
It is checked numerically, for two random choices of the parameters, that $A' = p\,A^*$: this follows from the reality of $|\mathcal{A}|^2$
and is therefore a consistency check of the whole previous calculation, not a choice.

In [7]:
def partner(DDD):
    new, p = sp.S(1), 1
    for D in DDD.as_ordered_factors():
        J, M, K = D.args[:3]
        new *= rot.D(J, -M, -K, *D.args[3:])
        p *= (-1) ** int(M - K)
    return new, p

blocks = []          # ('self', DDD) oppure ('pair', DDD, p)
used = set()
max_dev = 0.0
for DDD in angular_basis:
    if DDD in used:
        continue
    P, p = partner(DDD)
    if P == DDD:
        assert p == 1
        used.add(DDD)
        blocks.append(('self', DDD))
        for num in (num1, num2):
            max_dev = max(max_dev, abs(num[DDD].imag) / max(abs(num[DDD]), 1e-300))
        continue
    if P not in angular_trig_coefficients:
        raise ValueError(f"Partner mancante per {DDD}")
    used |= {DDD, P}
    blocks.append(('pair', DDD, p))
    for num in (num1, num2):
        A, Ap = num[DDD], num[P]
        max_dev = max(max_dev, abs(Ap - p * A.conjugate()) / max(abs(A), 1e-300))

n_self = sum(b[0] == 'self' for b in blocks)
n_pair = sum(b[0] == 'pair' for b in blocks)
print(f"Prodotti già reali (D = D'): {n_self}")
print(f"Coppie (D, D'):              {n_pair}")
print(f"Massima deviazione relativa da A' = p A* (e da Im A = 0 per i prodotti reali): {max_dev:.2e}")
assert max_dev < 1e-10

Prodotti già reali (D = D'): 75
Coppie (D, D'):              193
Massima deviazione relativa da A' = p A* (e da Im A = 0 per i prodotti reali): 1.58e-14


### Moments in explicitly real form
Each term of the coefficient $A$ is written as $c\,r$, with $c$ a numerical constant and $r$ a product of parameters.
From $r$ the real factors ($B_L$, $q$) and the pairs $z\,z^*\to|z|^2$ are separated; what remains, $\rho$, is a complex product
reduced to a canonical form chosen between $\rho$ and $\rho^*$.
After collecting, $\mathrm{Re}A$ and $\mathrm{Im}A$ become sums of terms of the form $\mathrm{Re}(\rho)$ and $\mathrm{Im}(\rho)$, as in Table II of Dey.

The moment associated with $\mathrm{Re}\,\mathcal{D}$ is $2\,\mathrm{Re}A$, the one associated with $\mathrm{Im}\,\mathcal{D}$ is $-2\,\mathrm{Im}A$,
and the one associated with an already real product is $\mathrm{Re}A$.

In [8]:
def is_real_factor(f):
    if f.is_number:
        return bool(f.is_real)
    return sp.conjugate(f) == f

def split_term(t):
    # t = c * real_part * rho, con c numero, real_part reale, rho complesso (o 1)
    c, real_part, rest = sp.S(1), [], []
    for f in sp.Mul.make_args(t):
        if f.is_number:
            c *= f
        elif is_real_factor(f):
            real_part.append(f)
        else:
            rest.append(f)
    # coppie z, conj(z) -> |z|^2
    rho = []
    pool = list(rest)
    while pool:
        f = pool.pop(0)
        fc = sp.conjugate(f)
        if fc in pool:
            pool.remove(fc)
            base = f.args[0] if isinstance(f, sp.conjugate) else f
            real_part.append(sp.Abs(base) ** 2)
        else:
            rho.append(f)
    return c, sp.Mul(*real_part), sp.Mul(*rho)

def real_imag_parts(A):
    # Restituisce (Re A, Im A) come espressioni esplicitamente reali
    acc = {}   # (real_part, rho_canonico) -> [X, Y]: A contiene X*rp*rho + Y*rp*conj(rho)
    for t in sp.Add.make_args(sp.expand(A)):
        c, rp, rho = split_term(t)
        rhoc = sp.conjugate(rho)
        if rho == rhoc:                      # termine reale
            key, slot = (rp, sp.S(1)), 0
            c = c * rho
        elif sp.default_sort_key(rho) <= sp.default_sort_key(rhoc):
            key, slot = (rp, rho), 0
        else:
            key, slot = (rp, rhoc), 1
        acc.setdefault(key, [sp.S(0), sp.S(0)])[slot] += c
    re_terms, im_terms = [], []
    for (rp, rho), (X, Y) in acc.items():
        if rho == 1:
            Xs = X
            re_terms.append(sp.re(Xs) * rp)
            im_terms.append(sp.im(Xs) * rp)
            continue
        # X rho + Y rho* : Re = Re((X+Y*) rho), Im = Im((X-Y*) rho)
        U = sp.expand(X + sp.conjugate(Y))
        W = sp.expand(X - sp.conjugate(Y))
        Rr, Ir = sp.re(rho, evaluate=False), sp.im(rho, evaluate=False)
        re_terms.append(rp * (sp.re(U) * Rr - sp.im(U) * Ir))
        im_terms.append(rp * (sp.re(W) * Ir + sp.im(W) * Rr))
    return sp.Add(*re_terms), sp.Add(*im_terms)

functions = []   # lista di dizionari: chiave = funzione angolare, momento = espressione reale
for b in tqdm(blocks, desc="Momenti reali"):
    DDD = b[1]
    ReA, ImA = real_imag_parts(angular_trig_coefficients[DDD])
    if b[0] == 'self':
        functions.append(dict(D=DDD, part='self', moment=ReA, num1=num1[DDD].real, num2=num2[DDD].real))
    else:
        functions.append(dict(D=DDD, part='re', moment=2 * ReA,
                              num1=2 * num1[DDD].real, num2=2 * num2[DDD].real))
        functions.append(dict(D=DDD, part='im', moment=-2 * ImA,
                              num1=-2 * num1[DDD].imag, num2=-2 * num2[DDD].imag))
print(f"Funzioni reali candidate: {len(functions)}")

Momenti reali:   0%|          | 0/268 [00:00<?, ?it/s]

Funzioni reali candidate: 461


### Checks on the moments
1. Each symbolic moment must coincide numerically with $2\mathrm{Re}A$, $-2\mathrm{Im}A$ or $A$ computed directly.
2. Identically vanishing moments (zero for both random choices of the parameters) are discarded.

In [9]:
max_rel = 0.0
for F in tqdm(functions, desc="Controllo"):
    for V, ref in ((V1, F['num1']), (V2, F['num2'])):
        val = fval(F['moment'], V)
        assert abs(val.imag) < 1e-10 * max(1, abs(val)), "momento non reale!"
        max_rel = max(max_rel, abs(val.real - ref) / max(abs(ref), 1e-12))
print(f"Massimo scarto relativo tra momento simbolico e valore diretto: {max_rel:.2e}")
assert max_rel < 1e-8

scale = max(abs(F['num1']) for F in functions)
is_zero = [abs(F['num1']) < 1e-12 * scale and abs(F['num2']) < 1e-12 * scale for F in functions]
zero = [F for F, z in zip(functions, is_zero) if z]
functions = [F for F, z in zip(functions, is_zero) if not z]
print(f"Momenti identicamente nulli scartati: {len(zero)}")

Controllo:   0%|          | 0/461 [00:00<?, ?it/s]

Massimo scarto relativo tra momento simbolico e valore diretto: 2.65e-13
Momenti identicamente nulli scartati: 0


### Angular functions
The functions are kept in Wigner form: $\mathrm{Re}(\mathcal{D})$, $\mathrm{Im}(\mathcal{D})$, or $\mathcal{D}$ if already real.
They are sorted by the rank of the three factors ($j_\ell$, $j_1$, $j_2$) and then by indices.

In [10]:
def ranks(DDD):
    out = {}
    for D in DDD.as_ordered_factors():
        J, M, K = D.args[:3]
        ang = D.args[3:]
        g = 'l' if thetal in ang else ('1' if theta1 in ang else '2')
        out[g] = (int(J), int(M), int(K))
    return out

def sort_key(F):
    r = ranks(F['D'])
    return (r['l'][0], r['1'][0], r['2'][0], r['l'][1:], r['1'][1:], r['2'][1:], {'self': 0, 're': 1, 'im': 2}[F['part']])

def angular_function(F):
    r = ranks(F['D'])
    D = (rot.D(*r['l'], 0, thetal, 0) * rot.D(*r['1'], phi, theta1, 0) * rot.D(*r['2'], chi, theta2, 0))
    if F['part'] == 're':
        return sp.re(D, evaluate=False)
    if F['part'] == 'im':
        return sp.im(D, evaluate=False)
    return D

# Se tutti gli indici M sono nulli, D non dipende dagli azimut ed è reale anche se K != 0:
# in quel caso Im(D) è identicamente nulla come funzione e va tolta dalla base.
# Succede solo per prodotti con indice leptonico k != 0, generati dal termine non risonante.
null_fun = [F for F in functions if F['part'] == 'im' and all(v[1] == 0 for v in ranks(F['D']).values())]
print(f"Funzioni Im(D) identicamente nulle tolte dalla base: {len(null_fun)}")
functions = [F for F in functions if not (F['part'] == 'im' and all(v[1] == 0 for v in ranks(F['D']).values()))]

functions.sort(key=sort_key)
for i, F in enumerate(functions, 1):
    F['index'] = i
    F['r'] = ranks(F['D'])
    F['f'] = angular_function(F)
print(f"Numero di funzioni angolari reali con momento non nullo: {len(functions)}")
print(f"  di cui già reali:      {sum(F['part']=='self' for F in functions)}")
print(f"  di cui parte reale:    {sum(F['part']=='re' for F in functions)}")
print(f"  di cui parte immag.:   {sum(F['part']=='im' for F in functions)}")

Funzioni Im(D) identicamente nulle tolte dalla base: 0
Numero di funzioni angolari reali con momento non nullo: 461
  di cui già reali:      75
  di cui parte reale:    193
  di cui parte immag.:   193


### Check against the squared amplitude
$\sum_i G_i\,f_i(\Omega)$ is compared with the saved squared amplitude `Atotsq` at random points of the angles and parameters.
If the reconstruction is correct the two values coincide.

In [11]:
rng = random.Random(7)
recon_err = 0.0
for V in (V1, V2):
    for _ in range(3):
        ang = {phi: rng.uniform(0, 2*np.pi), chi: rng.uniform(0, 2*np.pi),
               theta1: rng.uniform(0, np.pi), theta2: rng.uniform(0, np.pi), thetal: rng.uniform(0, np.pi)}
        direct = fval(Atotsq, V, ang)
        recon = sum(fval(F['moment'], V) * fval(F['f'], V, ang) for F in functions)
        print(f"|A|^2 diretto = {direct.real: .10e} (Im {direct.imag:.1e})   ricostruito = {recon.real: .10e}   scarto rel. = {abs(direct-recon)/abs(direct):.1e}")
        recon_err = max(recon_err, abs(direct - recon) / abs(direct))

|A|^2 diretto =  4.8592535710e-01 (Im -6.9e-17)   ricostruito =  4.8592535710e-01   scarto rel. = 5.5e-15


|A|^2 diretto =  3.9765799603e-01 (Im 1.0e-17)   ricostruito =  3.9765799603e-01   scarto rel. = 1.3e-14


|A|^2 diretto =  6.3143397681e-01 (Im 5.9e-17)   ricostruito =  6.3143397681e-01   scarto rel. = 4.7e-15


|A|^2 diretto =  2.1345908180e-03 (Im -1.3e-17)   ricostruito =  2.1345908180e-03   scarto rel. = 5.7e-14


|A|^2 diretto =  2.0741325299e-02 (Im -1.4e-17)   ricostruito =  2.0741325299e-02   scarto rel. = 2.7e-14


|A|^2 diretto =  9.6917244439e-02 (Im -1.5e-17)   ricostruito =  9.6917244439e-02   scarto rel. = 2.5e-14


### L/R separation and symmetry $\omega_i$
In the massless limit $G_i = g_i^L + \omega_i\, g_i^R$, as in Dey (eq. 34) and in the thesis (eq. 6.26).
The terms with chirality $\eta=-1$ and $\eta=+1$ are separated by reading the first argument of $H$, $\bar H$, $G$.
After the correction of the non-resonant term each term contains hadronic amplitudes of a single chirality; terms without $\eta$, if any, would be split in half.
$\omega_i$ is obtained by comparing **numerically** $g^R_i$ with $g^L_i$ after the replacement $\eta=-1\to+1$.

In [12]:
def flip_eta(expr):
    rep = {a: a.func(-a.args[0], *a.args[1:]) for a in expr.atoms(AppliedUndef) if a.func.__name__ in HADR}
    return expr.xreplace(rep)

for F in tqdm(functions, desc="L/R"):
    gL, gR = [], []
    for t in sp.Add.make_args(sp.expand(F['moment'])):
        e = term_eta(t)
        if e == -1:
            gL.append(t)
        elif e == 1:
            gR.append(t)
        else:
            gL.append(t / 2); gR.append(t / 2)
    F['gL'], F['gR'] = sp.Add(*gL), sp.Add(*gR)
    # Per il confronto servono valori per i parametri con eta=+1: si usano quelli già estratti
    w = None
    vals = []
    for V in (V1, V2):
        a = fval(flip_eta(F['gL']), V).real
        b = fval(F['gR'], V).real
        vals.append((a, b))
    if all(abs(b - a) < 1e-9 * max(abs(a), abs(b), 1e-30) for a, b in vals):
        w = 1
    elif all(abs(b + a) < 1e-9 * max(abs(a), abs(b), 1e-30) for a, b in vals):
        w = -1
    F['omega'] = w

from collections import Counter
print(Counter(F['omega'] for F in functions))

L/R:   0%|          | 0/461 [00:00<?, ?it/s]

Counter({1: 296, -1: 165})


### Orthogonality of the basis
The Gram matrix is computed with the scalar product of eq. 6.16 of the thesis.
Each function is $P(\theta_\ell,\theta_1,\theta_2)$ times $\cos(m_1\phi+m_2\chi)$ or $\sin(m_1\phi+m_2\chi)$,
so the azimuthal integral is analytic and the polar ones are computed with Gauss-Legendre quadrature in $\theta$.
This check does **not** correct anything: it only shows which functions, if any, are not orthogonal.

In [13]:
xg, wg = np.polynomial.legendre.leggauss(80)
th = 0.5 * np.pi * (xg + 1); wth = 0.5 * np.pi * wg * np.sin(th)   # misura d(cos theta)

_dcache = {}
def dvals(j, m, k):
    key = (j, m, k)
    if key not in _dcache:
        b = sp.Symbol('b')
        fn = sp.lambdify(b, rot.d(j, m, k, b).doit(), 'numpy')
        _dcache[key] = np.broadcast_to(np.asarray(fn(th), dtype=float), th.shape)
    return _dcache[key]

def polar(F, G, g):
    rf, rg = F['r'][g], G['r'][g]
    return np.sum(wth * dvals(*rf) * dvals(*rg))

def azim(F, G):
    # (1/4pi^2) * integrale su phi, chi
    a = (F['r']['1'][1], F['r']['2'][1]); b = (G['r']['1'][1], G['r']['2'][1])
    tf = 'cos' if F['part'] in ('re', 'self') else 'sin'
    tg = 'cos' if G['part'] in ('re', 'self') else 'sin'
    sf = -1 if F['part'] == 'im' else 1          # Im(e^{-ix}) = -sin x
    sg = -1 if G['part'] == 'im' else 1
    na = (-a[0], -a[1])
    if tf != tg:
        return 0.0
    if a == (0, 0) and b == (0, 0):
        return 1.0 * sf * sg if tf == 'cos' else 0.0
    same, opp = (a == b), (na == b)
    val = 0.5 * same + (0.5 if tf == 'cos' else -0.5) * opp
    return sf * sg * val

n = len(functions)
Gram = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        az = azim(functions[i], functions[j])
        if az == 0.0:
            continue
        v = az * polar(functions[i], functions[j], 'l') * polar(functions[i], functions[j], '1') * polar(functions[i], functions[j], '2') / 8
        Gram[i, j] = Gram[j, i] = v

norms = np.sqrt(np.diag(Gram))
Corr = Gram / np.outer(norms, norms)
off = [(i, j, Corr[i, j]) for i in range(n) for j in range(i+1, n) if abs(Corr[i, j]) > 1e-8]
print(f"Coppie non ortogonali: {len(off)}")
bad = sorted({functions[i]['index'] for i, j, _ in off} | {functions[j]['index'] for i, j, _ in off})
print(f"Funzioni coinvolte: {len(bad)}")
lep_k = [F['index'] for F in functions if ranks(F['D'])['l'][2] != 0]
print(f"Funzioni con indice leptonico k != 0: {len(lep_k)}")
print(f"Le funzioni non ortogonali hanno tutte k_l != 0 oppure sono accoppiate a una con k_l != 0: "
      f"{all(ranks(functions[i]['D'])['l'][2] != 0 or ranks(functions[j]['D'])['l'][2] != 0 for i, j, _ in off)}")

Coppie non ortogonali: 0
Funzioni coinvolte: 0
Funzioni con indice leptonico k != 0: 0
Le funzioni non ortogonali hanno tutte k_l != 0 oppure sono accoppiate a una con k_l != 0: True


### Independent functions
The angular functions are orthogonal, so as functions they are all linearly independent.
The redundancy lies in the **moments**: some $G_k$ are fixed linear combinations of others,
$G_k = \sum_j c_{kj}\, G_j$, for any value of the dynamical parameters ($a_i$, $H$, $\bar H$, $G$, $R$, $B_L$, $q$).
In that case the term $G_k f_k$ carries no new information and can be absorbed:
$$\frac{d\Gamma}{d\Omega}\propto\sum_{j\in\text{indep.}} G_j\, \tilde f_j,\qquad \tilde f_j = f_j + \sum_{k\in\text{dep.}} c_{kj}\, f_k .$$

**Method.** All moments are evaluated at many random points of the parameter space (more points than functions) and the matrix
$M_{ri} = G_i(\text{point } r)$ is built. The columns are examined in order with Gram-Schmidt: a column is independent if it is not
a combination of the previous ones. The order gives priority to functions with leptonic index $k=0$ and, among those, to lower indices,
so that the selected set is as "simple" as possible.
For each dependent moment the coefficients $c_{kj}$ are obtained by least squares and verified on new points.

Independence is meant for generic, independent dynamical parameters: additional relations imposed by the physics
(for example the LEET form factors) could reduce the number further.
The functions $\tilde f_j$ are in general no longer orthogonal to each other: this is quantified at the end of the cell with the Gram matrix.

In [14]:
# --- 1. Compilazione dei coefficienti A in forma numerica vettoriale ---
atom_list = sorted(dyn_atoms, key=str)
aidx = {a: i for i, a in enumerate(atom_list)}

def compile_factor(f):
    # restituisce (indice atomo, coniugato?, potenza) oppure None se f è un numero
    p = 1
    if f.is_Pow:
        f, p = f.base, int(f.exp)
    conj = False
    if isinstance(f, sp.conjugate):
        f, conj = f.args[0], True
    return aidx[f], conj, p

def compile_coeff(A):
    terms = []
    for t in sp.Add.make_args(A):
        c, facs = 1.0 + 0j, []
        for f in sp.Mul.make_args(t):
            if f.is_number:
                c *= fval(f, {})          # numeri con cache
            else:
                facs.append(compile_factor(f))
        terms.append((c, facs))
    return terms

block_of = {}           # D -> coefficiente compilato (uno per prodotto complesso)
for F in functions:
    if F['D'] not in block_of:
        block_of[F['D']] = compile_coeff(angular_trig_coefficients[F['D']])

def sample_matrix(n_draws, seed):
    # Matrice M[r, i] = G_i nel punto r dello spazio dei parametri
    rng = np.random.default_rng(seed)
    Vm = np.empty((len(atom_list), n_draws), dtype=complex)
    for i, a in enumerate(atom_list):
        if is_real_param(a):
            Vm[i] = rng.uniform(0.5, 1.5, n_draws)
        else:
            Vm[i] = rng.uniform(-1, 1, n_draws) + 1j * rng.uniform(-1, 1, n_draws)
    Vc = Vm.conj()
    Aval = {}
    for D, terms in block_of.items():
        acc = np.zeros(n_draws, dtype=complex)
        for c, facs in terms:
            v = np.full(n_draws, c)
            for i, conj, p in facs:
                x = Vc[i] if conj else Vm[i]
                v = v * (x if p == 1 else x ** p)
            acc += v
        Aval[D] = acc
    M = np.empty((n_draws, len(functions)))
    for col, F in enumerate(functions):
        A = Aval[F['D']]
        M[:, col] = {'self': A.real, 're': 2 * A.real, 'im': -2 * A.imag}[F['part']]
    return M

t0 = time.time()
n_draws = 2 * len(functions) + 100
M = sample_matrix(n_draws, seed=11)
print(f"Matrice dei momenti: {M.shape}, calcolata in {time.time()-t0:.1f} s")

# Controllo di coerenza con i valori calcolati prima (stesso valutatore su punti diversi: si confronta solo la scala)
assert np.all(np.isfinite(M))

# --- 2. Selezione greedy con Gram-Schmidt modificato ---
order = sorted(range(len(functions)), key=lambda i: (functions[i]['r']['l'][2] != 0, functions[i]['index']))
Mn = M / np.linalg.norm(M, axis=0)
TOL = 1e-8
Q = np.zeros((n_draws, 0))
indep, dep = [], []
for i in order:
    v = Mn[:, i].copy()
    for _ in range(2):                      # doppia ortogonalizzazione per stabilità
        v -= Q @ (Q.T @ v)
    nv = np.linalg.norm(v)
    if nv > TOL:
        indep.append(i)
        Q = np.column_stack([Q, v / nv])
    else:
        dep.append(i)

sv = np.linalg.svd(Mn, compute_uv=False)
print(f"Rango numerico (greedy): {len(indep)}")
print(f"Valori singolari attorno al salto: {sv[len(indep)-2:len(indep)+2]}")

# --- 3. Relazioni lineari per i momenti dipendenti, verificate su punti nuovi ---
M_test = sample_matrix(60, seed=12)
indep_sorted = sorted(indep, key=lambda i: functions[i]['index'])
Mi, Mi_test = M[:, indep_sorted], M_test[:, indep_sorted]
relations = {}
max_res = 0.0
for k in dep:
    c, *_ = np.linalg.lstsq(Mi, M[:, k], rcond=None)
    c[np.abs(c) < 1e-9 * np.abs(c).max()] = 0.0
    res = np.linalg.norm(Mi_test @ c - M_test[:, k]) / np.linalg.norm(M_test[:, k])
    max_res = max(max_res, res)
    relations[k] = {indep_sorted[j]: c[j] for j in np.nonzero(c)[0]}
print(f"Massimo residuo relativo delle relazioni su punti nuovi: {max_res:.1e}")
assert max_res < 1e-6

from fractions import Fraction
def exact_guess(x):
    # i coefficienti vengono da prodotti di Clebsch-Gordan: si cerca la forma ±sqrt(razionale)
    # si prende il denominatore più piccolo compatibile con la precisione numerica
    for lim in (10, 100, 1000):
        f = Fraction(x * x).limit_denominator(lim)
        g = sp.sign(x) * sp.sqrt(sp.Rational(f.numerator, f.denominator))
        if abs(float(g) - x) < 1e-10 * max(1, abs(x)):
            return g
    return sp.Float(x, 12)

for F in functions:
    F['independent'] = None
    F['relation'] = None
for i in indep:
    functions[i]['independent'] = True
for k, rel in relations.items():
    functions[k]['independent'] = False
    functions[k]['relation'] = {functions[j]['index']: exact_guess(c) for j, c in rel.items()}

# --- 4. Funzioni efficaci f~_j = f_j + sum_k c_kj f_k e loro ortogonalità ---
Cmat = np.zeros((len(functions), len(indep_sorted)))      # f~ = F^T C
for col, j in enumerate(indep_sorted):
    Cmat[j, col] = 1.0
for k, rel in relations.items():
    for j, c in rel.items():
        Cmat[k, indep_sorted.index(j)] += c
for col, j in enumerate(indep_sorted):
    F = functions[j]
    F['f_eff'] = F['f'] + sum(sp.Float(Cmat[k, col], 12) * functions[k]['f']
                              for k in range(len(functions)) if k != j and Cmat[k, col] != 0)
G_eff = Cmat.T @ Gram @ Cmat
d = np.sqrt(np.clip(np.diag(G_eff), 1e-300, None))
Corr_eff = G_eff / np.outer(d, d)
np.fill_diagonal(Corr_eff, 0)

phys_ind = [functions[i] for i in indep if functions[i]['r']['l'][2] == 0]
print()
print(f"Funzioni totali: {len(functions)}")
print(f"  momenti indipendenti: {len(indep)}  (di cui con k_l = 0: {len(phys_ind)})")
print(f"  momenti dipendenti:   {len(dep)}")
print(f"  simmetria tra gli indipendenti: +1 -> {sum(functions[i]['omega']==1 for i in indep)}, "
      f"-1 -> {sum(functions[i]['omega']==-1 for i in indep)}, non definita -> {sum(functions[i]['omega'] is None for i in indep)}")
n_exact = sum(not isinstance(c, sp.Float) for F in functions if F['relation'] for c in F['relation'].values())
n_coef = sum(len(F['relation']) for F in functions if F['relation'])
print(f"Coefficienti c_kj riconosciuti come ±sqrt(razionale): {n_exact}/{n_coef}")
up = np.abs(np.triu(Corr_eff))
print(f"Funzioni efficaci f~ non ortogonali tra loro: {int(np.sum(up > 1e-8))} coppie su {len(indep)*(len(indep)-1)//2}")
print(f"  con |correlazione| > 0.5: {int(np.sum(up > 0.5))},  > 0.9: {int(np.sum(up > 0.9))},  massima: {up.max():.6f}")
ii, jj = np.unravel_index(np.argmax(up), up.shape)
print(f"  coppia più correlata: f~_{functions[indep_sorted[ii]]['index']} e f~_{functions[indep_sorted[jj]]['index']}")

print("\nEsempi di relazioni:")
for k in sorted(relations, key=lambda k: functions[k]['index'])[:8]:
    rel = functions[k]['relation']
    rhs = " + ".join(f"({v})·G_{j}" for j, v in rel.items())
    print(f"  G_{functions[k]['index']} = {rhs}")

Matrice dei momenti: (1022, 461), calcolata in 1.9 s
Rango numerico (greedy): 340
Valori singolari attorno al salto: [1.65544093e-02 5.46812038e-03 1.30567060e-15 1.25975976e-15]


Massimo residuo relativo delle relazioni su punti nuovi: 1.8e-13

Funzioni totali: 461
  momenti indipendenti: 340  (di cui con k_l = 0: 340)
  momenti dipendenti:   121
  simmetria tra gli indipendenti: +1 -> 215, -1 -> 125, non definita -> 0
Coefficienti c_kj riconosciuti come ±sqrt(razionale): 299/336
Funzioni efficaci f~ non ortogonali tra loro: 373 coppie su 57630
  con |correlazione| > 0.5: 95,  > 0.9: 15,  massima: 0.999463
  coppia più correlata: f~_14 e f~_27

Esempi di relazioni:
  G_31 = (10/7)·G_5 + (-15/7)·G_16
  G_42 = (-20*sqrt(3)/3)·G_4 + (22*sqrt(6)/3)·G_14 + (14*sqrt(2))·G_27 + (13.9664546519)·G_29
  G_44 = (4)·G_16
  G_54 = (48/7)·G_4 + (-5*sqrt(2))·G_14 + (-3*sqrt(6))·G_27 + (-6.91160258818)·G_29
  G_55 = (sqrt(10))·G_14 + (3*sqrt(30)/5)·G_27
  G_57 = (-8*sqrt(15)/15)·G_29
  G_58 = (-8*sqrt(15)/15)·G_30
  G_59 = (18/7)·G_5 + (-20/7)·G_16


### Summary and saving
- `REIM_moments.pkl`: all functions, with the fields `independent` and `relation` (for dependent ones: $G_k=\sum_j c_{kj}G_j$, keys = indices $j$).
- `REIM_moments_independent.pkl`: only the independent functions, with the effective function `f_eff` $=\tilde f_j$.
- Folder `Moments_ReIm`: one file `D_<index>_omega_<piu|meno|misto>.txt` for each **independent** function
  (angular function, effective function, $\omega_i$, moment $g_i^L$), where `piu`, `meno` and `misto` stand for $\omega=+1$, $-1$ and undefined,
  and the file `relazioni.txt` with the relations of the dependent moments.

In [15]:
records = []
for F in functions:
    records.append(dict(index=F['index'], f=F['f'], D=F['D'], part=F['part'],
                        omega=F['omega'], gL=F['gL'], gR=F['gR'], moment=F['moment'],
                        independent=F['independent'], relation=F['relation'],
                        f_eff=F.get('f_eff')))
with open("REIM_moments.pkl", "wb") as fh:
    dill.dump(records, fh)
records_ind = [R for R in records if R['independent']]
with open("REIM_moments_independent.pkl", "wb") as fh:
    dill.dump(records_ind, fh)

for old in os.listdir(OUTDIR):
    if old.endswith(".txt"):
        os.remove(os.path.join(OUTDIR, old))
for R in records_ind:
    tag = {1: 'piu', -1: 'meno', None: 'misto'}[R['omega']]
    with open(os.path.join(OUTDIR, f"D_{R['index']:03d}_omega_{tag}.txt"), "w", encoding="utf-8") as fh:
        fh.write(f"D_{R['index']} = {R['f']}\n")
        fh.write(f"f_eff_{R['index']} = {R['f_eff']}\n")
        fh.write(f"omega = {R['omega']}\n\n")
        fh.write("g^L =\n")
        fh.write(str(R['gL']) + "\n")
with open(os.path.join(OUTDIR, "relazioni.txt"), "w", encoding="utf-8") as fh:
    for R in records:
        if R['independent'] is False:
            rhs = " + ".join(f"({v})*G_{j}" for j, v in R['relation'].items())
            fh.write(f"G_{R['index']} = {rhs}    [D_{R['index']} = {R['f']}]\n")

print(f"Funzioni angolari totali: {len(records)}")
print(f"Funzioni indipendenti salvate: {len(records_ind)}")
for w, name in ((1, '+(L->R)'), (-1, '-(L->R)'), (None, 'non definita')):
    print(f"  simmetria {name}: {sum(R['omega'] == w for R in records_ind)}")

phys = [F for F in functions if F['r']['l'][2] == 0]
art = [F for F in functions if F['r']['l'][2] != 0]
print()
print(f"Funzioni con indice leptonico k = 0: {len(phys)} (indipendenti: {sum(bool(F['independent']) for F in phys)})")
print(f"Funzioni con indice leptonico k != 0: {len(art)} (indipendenti: {sum(bool(F['independent']) for F in art)})")

Funzioni angolari totali: 461
Funzioni indipendenti salvate: 340
  simmetria +(L->R): 215
  simmetria -(L->R): 125
  simmetria non definita: 0

Funzioni con indice leptonico k = 0: 461 (indipendenti: 340)
Funzioni con indice leptonico k != 0: 0 (indipendenti: 0)


### Examples: the first functions and their moments $g^L$

In [16]:
for R in records[:3]:
    print(f"D_{R['index']}  (omega = {R['omega']})")
    sp.pprint(R['f'])
    print(f"numero di termini in g^L: {len(sp.Add.make_args(R['gL']))}")
    print()
# una funzione Re e la sua Im
R_re = next(R for R in records if R['part'] == 're')
R_im = next(R for R in records if R['part'] == 'im' and R['D'] == R_re['D'])
for R in (R_re, R_im):
    print(f"D_{R['index']}  (omega = {R['omega']})")
    sp.pprint(R['f'])
    sp.pprint(R['gL'])
    print()

D_1  (omega = 1)
 0            0            0          
d   (thetal)⋅D   (χ,θ₂,0)⋅D   (φ,θ₁,0)
 0,0          0,0          0,0        
numero di termini in g^L: 422

D_2  (omega = 1)
 0            0            1          
d   (thetal)⋅D   (φ,θ₁,0)⋅D   (χ,θ₂,0)
 0,0          0,0          0,0        
numero di termini in g^L: 111

D_3  (omega = 1)
 0            0            2          
d   (thetal)⋅D   (φ,θ₁,0)⋅D   (χ,θ₂,0)
 0,0          0,0          0,0        
numero di termini in g^L: 406

D_8  (omega = 1)
  ⎛ 0            1            1          ⎞
re⎜d   (thetal)⋅D   (φ,θ₁,0)⋅D   (χ,θ₂,0)⎟
  ⎝ 0,0          0,1          1,0        ⎠


                         2                                                              ⎛                                                     __ _____________ ↪
  0.0833333333333333⋅√6⋅q ⋅BW(1, r₂, K*(1410))⋅BW(1, r₂, K*(892))⋅BW(1, r₃, K1(1650))⋅re⎝a₂₃⋅H̅(-1, 1, -1, 0, K*(1410))⋅R(K*(1410))⋅R(K*(892))⋅a₃⋅H(-1, 1, -1,  ↪
- ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                                            2                                                                  ↪
                                                                                           π                                                                   ↪

↪ ____________ ___________⎞       2                                                              ⎛                                                    __ _____ ↪
↪ 0, K1(1650))⋅R(K1(1650))⎠   √6

                       2                                                              ⎛                                                     __ _______________ ↪
0.0833333333333333⋅√6⋅q ⋅BW(1, r₂, K*(1410))⋅BW(1, r₂, K*(892))⋅BW(1, r₃, K1(1650))⋅im⎝a₂₃⋅H̅(-1, 1, -1, 0, K*(1410))⋅R(K*(1410))⋅R(K*(892))⋅a₃⋅H(-1, 1, -1, 0, ↪
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                                          2                                                                    ↪
                                                                                         π                                                                     ↪

↪ __________ ___________⎞       2                                                              ⎛                                                    __ _______ ↪
↪  K1(1650))⋅R(K1(1650))⎠   √6⋅q

### Detailed summary
All numbers are read from the variables computed above, not written by hand.
The order follows the priority of the moment analysis: first the **orthogonal basis** of angular functions, then the number of **independent moments**.

In [17]:
from collections import Counter

def line(s=""):
    print(s)

phys_idx = [n for n, F in enumerate(functions) if F['r']['l'][2] == 0]
art_idx  = [n for n, F in enumerate(functions) if F['r']['l'][2] != 0]
phys_set = set(phys_idx)
up_all = np.abs(np.triu(Corr, 1))
nonorth_phys = sum(1 for i, j, _ in off if i in phys_set and j in phys_set)
nonorth_mixed = sum(1 for i, j, _ in off if (i in phys_set) != (j in phys_set))
nonorth_art = sum(1 for i, j, _ in off if i not in phys_set and j not in phys_set)

line("=" * 78)
line("1. PUNTO DI PARTENZA (modulo quadro di AmpiezzaWigner.ipynb + correzione del non risonante)")
line("=" * 78)
line(f"   Termini nel modulo quadro salvato:                  {len(terms_old)}")
line(f"     di cui con a_NR (corretti con d^1_(0,eta)(theta_l)): {sum(1 for t in terms_old if t.has(aNR))}")
line(f"   Controllo NR vs canale K(1460) con R=1 (scarto max): {max_chk:.1e}")
line(f"   Termini nel modulo quadro corretto |A|^2:           {len(sp.Add.make_args(Atotsq))}")
line(f"   Prodotti complessi di tre Wigner D dopo la serie CG: {len(angular_basis)}")
line(f"     - prodotti già reali (D = D'):                    {n_self}")
line(f"     - coppie (D, D') con D* = p D':                   {n_pair}")
line(f"   Verifica A' = p A* sulle coppie (scarto rel. max):  {max_dev:.1e}")
line(f"   Vecchia base (solo Re D, coefficiente A + A'):      {n_self + n_pair} funzioni, informazione incompleta")

line()
line("=" * 78)
line("2. BASE ORTOGONALE DI FUNZIONI ANGOLARI REALI (forma di Wigner)")
line("=" * 78)
line(f"   Funzioni candidate: {n_self} (D reale) + {n_pair} (Re D) + {n_pair} (Im D) = {n_self + 2*n_pair}")
line(f"   Im D identicamente nulle (tutti gli M = 0), tolte:  {len(null_fun)}")
line(f"   Momenti identicamente nulli, tolti:                 {len(zero)}")
line(f"   FUNZIONI NELLA BASE:                                {len(functions)}")
cnt = Counter(F['part'] for F in functions)
line(f"     - D già reali: {cnt['self']},  Re D: {cnt['re']},  Im D: {cnt['im']}")
line()
line(f"   Indice leptonico k = 0 (d^j_(m,0)(theta_l)):        {len(phys_idx)}")
line(f"     coppie non ortogonali tra queste:                 {nonorth_phys}   -> base ortogonale")
w = Counter(functions[n]['omega'] for n in phys_idx)
line(f"     simmetria L->R: +1 -> {w[1]}, -1 -> {w[-1]}, non definita -> {w[None]}")
line(f"   Indice leptonico k != 0:                            {len(art_idx)}")
if art_idx:
    w = Counter(functions[n]['omega'] for n in art_idx)
    line(f"     simmetria L->R: +1 -> {w[1]}, -1 -> {w[-1]}, non definita -> {w[None]}")
    line(f"     coppie non ortogonali tra loro:                   {nonorth_art}")
    line(f"     coppie non ortogonali con funzioni k = 0:         {nonorth_mixed}")
line(f"   Totale coppie non ortogonali nella base:            {len(off)}")
line()
line(f"   Momenti simbolici vs valori diretti (scarto max):   {max_rel:.1e}")
line(f"   Ricostruzione di |A|^2 in punti casuali (scarto max): {recon_err:.1e}")

line()
line("=" * 78)
line("3. MOMENTI INDIPENDENTI")
line("=" * 78)
ind_phys = [i for i in indep if i in phys_set]
ind_art = [i for i in indep if i not in phys_set]
line(f"   Rango della matrice dei momenti ({M.shape[0]} punti x {M.shape[1]} funzioni): {len(indep)}")
line(f"   Salto nei valori singolari: {sv[len(indep)-1]:.1e} -> {sv[len(indep)]:.1e}")
line(f"   Momenti indipendenti tra le {len(phys_idx)} funzioni con k = 0: {len(ind_phys)}")
line(f"     (le funzioni k = 0 sono esaminate per prime: {len(ind_phys)} è il rango dei soli momenti k = 0)")
line(f"   Momenti indipendenti aggiunti dalle funzioni k != 0:  {len(ind_art)}")
line(f"   Momenti dipendenti (relazioni G_k = sum c_kj G_j):    {len(dep)}")
line(f"     di cui tra funzioni k = 0:                        {sum(1 for k in dep if k in phys_set)}")
line(f"   Residuo massimo delle relazioni su punti nuovi:     {max_res:.1e}")
line(f"   Coefficienti c_kj della forma ±sqrt(razionale):     {n_exact}/{n_coef} (gli altri sono somme di radici, lasciati numerici)")
w = Counter(functions[i]['omega'] for i in indep)
line(f"   Simmetria dei momenti indipendenti: +1 -> {w[1]}, -1 -> {w[-1]}, non definita -> {w[None]}")
line()
line("   Se si riassorbono i dipendenti nelle funzioni efficaci f~_j, l'ortogonalità si perde:")
line(f"     coppie f~ non ortogonali: {int(np.sum(up > 1e-8))},  |corr| > 0.9: {int(np.sum(up > 0.9))},  massima: {up.max():.6f}")

line()
line("=" * 78)
line("4. FILE SALVATI")
line("=" * 78)
line(f"   REIM_moments.pkl              tutte le {len(records)} funzioni (campi independent, relation)")
line(f"   REIM_moments_independent.pkl  le {len(records_ind)} funzioni con momento indipendente (con f_eff)")
line(f"   {os.path.basename(OUTDIR)}/                  {len(records_ind)} file .txt + relazioni.txt")

1. PUNTO DI PARTENZA (modulo quadro di AmpiezzaWigner.ipynb + correzione del non risonante)
   Termini nel modulo quadro salvato:                  33281


     di cui con a_NR (corretti con d^1_(0,eta)(theta_l)): 513
   Controllo NR vs canale K(1460) con R=1 (scarto max): 2.3e-15
   Termini nel modulo quadro corretto |A|^2:           33282
   Prodotti complessi di tre Wigner D dopo la serie CG: 461
     - prodotti già reali (D = D'):                    75
     - coppie (D, D') con D* = p D':                   193
   Verifica A' = p A* sulle coppie (scarto rel. max):  1.6e-14
   Vecchia base (solo Re D, coefficiente A + A'):      268 funzioni, informazione incompleta

2. BASE ORTOGONALE DI FUNZIONI ANGOLARI REALI (forma di Wigner)
   Funzioni candidate: 75 (D reale) + 193 (Re D) + 193 (Im D) = 461
   Im D identicamente nulle (tutti gli M = 0), tolte:  0
   Momenti identicamente nulli, tolti:                 0
   FUNZIONI NELLA BASE:                                461
     - D già reali: 75,  Re D: 193,  Im D: 193

   Indice leptonico k = 0 (d^j_(m,0)(theta_l)):        461
     coppie non ortogonali tra queste:                 0   -> base 